# Attacks on the dataset belonging to the paper "Dismantling real world ECC"

In [ ]:
import trsfile
import matplotlib.pyplot as plt
import numpy as np
from tqdm import trange, tqdm
import seaborn as sns
import bokeh.plotting as bk
import lpips
import librosa
import torchaudio
import torch
import time
import math

from skimage.metrics import structural_similarity as ssim
import torch
import torch.nn.functional as F
import numpy as np

from bokeh.io import output_notebook 
output_notebook()

In [2]:
def compute_correlation(target_trace, template_trace):
    corr = np.corrcoef(target_trace, template_trace)[0,1] # Take the non-identity part of the correlation matrix. Note that [0,1] = [1,0]
    return np.abs(corr)

In [3]:
def plot_similarity(similarity_array0, similarity_array1, iteration, similarity_function):
        plt.figure(figsize=(10,5))
        plt.plot(similarity_array0, label = "bit 0")
        plt.plot(similarity_array1, label = "bit 1")
        plt.title("Similarity for template 2P and 3P on {}".format(similarity_function.__name__))
        plt.xlabel("Timesamples")
        plt.ylabel("Delta power consumption")
        plt.legend()
        plt.savefig("./ota2_scalar1{}_{}.png".format(iteration, similarity_function.__name__), bbox_inches='tight')
        plt.show()

def pattern_match(target_trace, template_trace, comparison_function, poi_start, poi_end, step=1):
    comparison_array = []
    window = poi_end - poi_start
    template_trace_sliced = template_trace[poi_start:poi_end]
    for timesample in trange(0, len(target_trace) - window, step):
        comparison = comparison_function(target_trace[timesample:timesample + window], template_trace_sliced)
        comparison_array.append(comparison)
    return np.array(comparison_array)

def determine_keybit(comparison_array0, comparison_array1, sign):
    # Get the highest peak of both, and take that timesample as comparison point
    if sign == "max":
        peak0 = np.max(comparison_array0)
        peak1 = np.max(comparison_array1)
        keybit_guess = np.argmax([peak0, peak1]) 
    elif sign == "min":
        peak0 = np.min(comparison_array0)
        peak1 = np.min(comparison_array1)
        keybit_guess = np.argmin([peak0, peak1]) 
    
    return int(keybit_guess)

## Load in datasets

In [6]:
## Scalar  = 2 => 10 in binary
single_target_trace_sc2 = np.array(trsfile.open("../../datasets/ota2/P1_scalar2_unprotected_ec_scalmul.trs", 'r')[0], dtype=np.float64)
single_template_2P_sc2 = np.array(trsfile.open('../../datasets/ota2/2P1_scalar2_unprotected_ec_scalmul.trs', 'r')[0], dtype=np.float64)
single_template_3P_sc2 = np.array(trsfile.open('../../datasets/ota2/3P1_scalar2_unprotected_ec_scalmul.trs', 'r')[0], dtype=np.float64)

## Scalar  = 3 => 11 in binary
single_target_trace_sc3 = np.array(trsfile.open("../../datasets/ota2/P1_scalar3_unprotected_ec_scalmul.trs", 'r')[0], dtype = np.float64)
single_template_2P_sc3 = np.array(trsfile.open('../../datasets/ota2/2P1_scalar3_unprotected_ec_scalmul.trs', 'r')[0], dtype = np.float64)
single_template_3P_sc3 = np.array(trsfile.open('../../datasets/ota2/3P1_scalar3_unprotected_ec_scalmul.trs', 'r')[0], dtype = np.float64)


In [7]:
single_trace_leakage_sc2 = [single_target_trace_sc2, single_template_2P_sc2, single_template_3P_sc2]
single_trace_leakage_sc3 = [single_target_trace_sc3, single_template_2P_sc3, single_template_3P_sc3]

## Correlation-based attack single-trace attack 

In [ ]:
step_size = 10
poi_start = 1079000
poi_end = 1098000

for correct_keybit, single_trace in enumerate([single_trace_leakage_sc2, single_trace_leakage_sc3]):
    corr_results_template0 = pattern_match(single_trace[0], single_trace[1], compute_correlation, poi_start, poi_end, step=step_size)
    corr_results_template1 = pattern_match(single_trace[0], single_trace[2], compute_correlation, poi_start, poi_end, step=step_size)

    plot_similarity(corr_results_template0, corr_results_template1, correct_keybit, similarity_function=compute_correlation, leakage = "horizontal")
    keybit_guess = determine_keybit(corr_results_template0, corr_results_template1, sign="max")
    
    print("The correct keybit is: 1", correct_keybit)
    print("The found keybit is: 1" + str(keybit_guess))
    print("Current scalar value: " + str(1 << 1 | keybit_guess) + "P")

    np.savez("./ota2_sc1{}_{}".format(correct_keybit, compute_correlation.__name__),  comparison_0 = corr_results_template0, comparison_1 = corr_results_template1)


## Similarity based attacks (SSIM + LPIPS)

In [15]:
def calc_spectrogram(trace, n_fft = 512): #NOTE: it is very sensitive for this value
    trace_tensor = torch.tensor(trace, dtype=torch.float32)
    spec = torchaudio.transforms.Spectrogram(n_fft=n_fft)(trace_tensor)
    spec_db = librosa.power_to_db(spec.numpy(), ref=np.max)

    # Normalize to [0,1]
    spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)

    return spec_db

def spec_to_tensor(spec_db, target_size):
    # Convert to 3-channel float32 tensor for LPIPS: (C, H, W)
    spec_tensor = torch.tensor(spec_db).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    # Resize to target_size
    spec_resized = F.interpolate(spec_tensor, size=target_size, mode='bilinear', align_corners=False)

    return spec_resized

In [ ]:
## LPIPS model
device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_model = lpips.LPIPS(net='alex')
lpips_model.eval()
lpips_model.to(device)


In [17]:
def compute_distance_lpips(target_trace, template_trace):
    tensor_size = (224, 224)  # LPIPS expects 224x224 input
    
    target_spec  = calc_spectrogram(target_trace, 512)
    template_spec = calc_spectrogram(template_trace, 512)

    target_tensor = spec_to_tensor(target_spec, tensor_size).to(device)
    template_tensor = spec_to_tensor(template_spec, tensor_size).to(device)

    d = lpips_model.forward(target_tensor,template_tensor)
    return d.detach().cpu().numpy().ravel()[0]

def compute_structural_similarity(target_trace, template_trace):
    """Compute structural similarity between two traces."""
    target_tensor = calc_spectrogram(target_trace, 512)
    template_tensor = calc_spectrogram(template_trace, 512)

    return ssim(target_tensor, template_tensor, data_range=target_tensor.max() - target_tensor.min())

In [18]:
poi_start = 1079000
poi_end = 1098000
window = poi_end - poi_start
step_size = 10

In [ ]:
similarity_methods = [compute_structural_similarity, compute_distance_lpips]

for method in similarity_methods:
    for correct_keybit, single_trace_leakage in enumerate([single_trace_leakage_sc2, single_trace_leakage_sc3]):
        comparison_0 = pattern_match(single_trace_leakage[0], single_trace_leakage[1], method, poi_start, poi_end, step=step_size)
        comparison_1 = pattern_match(single_trace_leakage[0], single_trace_leakage[2], method, poi_start, poi_end, step=step_size)

        plot_similarity(comparison_0, comparison_1, correct_keybit, similarity_function=method, leakage = "horizontal")
        keybit_guess = determine_keybit(comparison_0, comparison_1, sign="max" if method != compute_distance_lpips else "min") 
        
        print("The correct keybit is: 1", correct_keybit)
        print("The found keybit using {} is: 1{}".format(method.__name__, keybit_guess))
        print("Current scalar value: " + str(1 << 1| keybit_guess) + "P")

        np.savez("./ota2_sc1{}_{}".format(correct_keybit, method.__name__),  comparison_0 = comparison_0, comparison_1 = comparison_1)
